# Explorador de Tablas

Ingresa el nombre de cualquier tabla de la base de datos y visualiza sus datos en formato tabla.

In [64]:
from pathlib import Path
import subprocess
import tempfile

import pandas as pd
from IPython.display import display, HTML

DB_PATH = "localhost:/Users/wilsonjonatan/Documents/bases 1 2026/violencia guate/sql/db/violencia_guate.fdb"
DB_USER = "sysdba"
DB_PASSWORD = "masterkey"
ISQL_PATH = "/Library/Frameworks/Firebird.framework/Resources/bin/isql"

In [ ]:
def run_query(sql: str) -> pd.DataFrame:
    query = sql.strip().rstrip(";") + ";"
    script = f"SET HEADING OFF;\nSET LIST ON;\n{query}\n"

    with tempfile.NamedTemporaryFile("w", suffix=".sql", delete=False, encoding="utf-8") as tmp:
        tmp.write(script)
        tmp_path = tmp.name

    try:
        result = subprocess.run(
            [ISQL_PATH, "-user", DB_USER, "-password", DB_PASSWORD, DB_PATH, "-q", "-i", tmp_path],
            capture_output=True,
            text=True,
        )
    finally:
        Path(tmp_path).unlink(missing_ok=True)

    output = "\n".join(part for part in (result.stdout, result.stderr) if part)
    rows = []
    current = {}

    for raw_line in output.splitlines():
        line = raw_line.strip()
        if not line:
            if current:
                rows.append(current)
                current = {}
            continue
        if line.startswith("SQL>") or line.startswith("DatabaseError") or line.startswith("Warning"):
            continue

        parts = line.split(None, 1)
        if len(parts) == 2:
            key, value = parts
            current[key.lower()] = value.strip()

    if current:
        rows.append(current)

    df = pd.DataFrame(rows)
    for column in df.columns:
        numeric = pd.to_numeric(df[column], errors="coerce")
        if len(df[column]) > 0 and numeric.notna().all():
            df[column] = numeric
    return df


def get_table_info(tabla: str) -> tuple:
    """Obtiene el conteo y primeras filas de una tabla."""
    sql_count = f"SELECT COUNT(*) AS total FROM {tabla};"
    sql_select = f"SELECT * FROM {tabla};"
    
    try:
        df_count = run_query(sql_count)
        if df_count.empty:
            return None, 0
        
        total = int(df_count.iloc[0, 0])
        df_data = run_query(sql_select)
        return df_data, total
    except Exception as e:
        print(f"Error: {e}")
        return None, 0

In [76]:
# ========== CONFIGURACIÓN: escribe aquí cualquier consulta SQL ==========
CONSULTA_SQL = """



 
SELECT
    de.nombre AS delito,
    COUNT(*) AS total_exhumaciones
FROM exhumacion ex
JOIN hecho h ON h.id_hecho = ex.id_hecho
LEFT JOIN hecho_delictivo hd ON hd.id_hecho = h.id_hecho
LEFT JOIN delito_cometido dc ON dc.id_delito_cometido = hd.id_delito
LEFT JOIN delito de ON de.id_delito = dc.id_tipo_delito
GROUP BY 1
ORDER BY total_exhumaciones DESC;



"""
# =======================================================================

print("\n📊 Ejecutando consulta:")
print(CONSULTA_SQL)
print("=" * 60)

try:
    df = run_query(CONSULTA_SQL)

    if df.empty:
        print("⚠️ La consulta no devolvió filas.")
    else:
        print(f"✓ Filas obtenidas: {len(df)}")
        print(f"✓ Columnas: {len(df.columns)}")
        print(f"\nEncabezados: {', '.join(df.columns)}")
        print("-" * 60)
        display(df.head(MAX_FILAS_MOSTRAR))

except Exception as e:
    print(f"❌ Error al ejecutar la consulta: {e}")


📊 Ejecutando consulta:





SELECT
    de.nombre AS delito,
    COUNT(*) AS total_exhumaciones
FROM exhumacion ex
JOIN hecho h ON h.id_hecho = ex.id_hecho
LEFT JOIN hecho_delictivo hd ON hd.id_hecho = h.id_hecho
LEFT JOIN delito_cometido dc ON dc.id_delito_cometido = hd.id_delito
LEFT JOIN delito de ON de.id_delito = dc.id_tipo_delito
GROUP BY 1
ORDER BY total_exhumaciones DESC;




✓ Filas obtenidas: 84
✓ Columnas: 2

Encabezados: delito, total_exhumaciones
------------------------------------------------------------


,delito,total_exhumaciones
0,Falsedad ideológica,4
1,Apropiación irregular,4
2,Contra los recursos forestales,4
3,Amenazas,3
4,Asesinato,3
...,...,...
79,Atentado contra el patrimonio natural y cultur...,1
80,Uso de equipos terminales móviles por funciona...,1
81,"Enfermedad vascular, no especificando si es he...",1
82,Portación ilegal de armas de fuego bélicas o d...,1
